<a href="https://colab.research.google.com/github/LaraDondossola/Classificacao-eventos-climaticos/blob/main/Notebooks/Interface-eventos-clim%C3%A1ticos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install -q streamlit joblib pandas scikit-learn
!npm install -g localtunnel
!pip install -q pyngrok streamlit

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
changed 22 packages in 2s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [31]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.base import BaseEstimator, TransformerMixin

# --- DEFINIÇÃO DA CLASSE CUSTOMIZADA PARA O JOBLIB ---
class OutlierCapper(BaseEstimator, TransformerMixin):
    """
    Classe customizada para clipping/winsorização de outliers.
    Necessária aqui para que o joblib consiga recriar o objeto preprocessor.
    """
    def __init__(self, lower_quantile=0.0, upper_quantile=0.99):
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile
        self.lower_bounds_ = None
        self.upper_bounds_ = None

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        self.lower_bounds_ = X_df.quantile(self.lower_quantile).values
        self.upper_bounds_ = X_df.quantile(self.upper_quantile).values
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        return X_df.clip(lower=self.lower_bounds_, upper=self.upper_bounds_, axis=1).values

# Configuração da página
st.set_page_config(
    page_title="Predição de Eventos Climáticos",
    page_icon="⛈️",
    layout="wide"
)

# --- CSS CUSTOMIZADO PARA OCULTAR OS BOTÕES DE INCREMENTO E DECREMENTO (- e +) ---
st.markdown("""
    <style>
    /* Oculta os botões do Streamlit */
    button[data-testid="stNumberInputStepDown"],
    button[data-testid="stNumberInputStepUp"] {
        display: none !important;
    }
    /* Oculta as setas nativas do navegador para inputs numéricos */
    input[type=number]::-webkit-inner-spin-button,
    input[type=number]::-webkit-outer-spin-button {
        -webkit-appearance: none;
        margin: 0;
    }
    input[type=number] {
        -moz-appearance: textfield;
    }
    </style>
""", unsafe_allow_html=True)

# Título e Descrição
st.title("⛈️ Painel de Avaliação de Impacto e Risco Climático")
st.markdown("Insira os dados da ocorrência abaixo para calcular o **Nível de Risco** e estimar a **População Afetada**.")

# --- CARREGAMENTO DOS MODELOS E PREPROCESSOR ---
@st.cache_resource
def carregar_artefatos():
    base_path = Path("Models") if Path("Models").exists() else Path(".")

    # Nomes dos arquivos ajustados para os novos pickles do KNN
    preprocessor = joblib.load(base_path / "preprocessor.pkl")
    modelo_clf = joblib.load(base_path / "modelo_b_classificacao_knn.pkl")
    modelo_reg = joblib.load(base_path / "modelo_1_regressao_knn.pkl")

    return preprocessor, modelo_clf, modelo_reg

try:
    preprocessor, modelo_clf, modelo_reg = carregar_artefatos()
    st.sidebar.success("✅ Modelos e Preprocessor carregados com sucesso!")
except Exception as e:
    st.error(f"Erro ao carregar os modelos: {e}")
    st.stop()

# --- FORMULÁRIO DE ENTRADA DE DADOS ---
st.header("📋 Dados da Ocorrência")

# Mapeamento de Meses
meses_map = {
    "Janeiro": 1, "Fevereiro": 2, "Março": 3, "Abril": 4,
    "Maio": 5, "Junho": 6, "Julho": 7, "Agosto": 8,
    "Setembro": 9, "Outubro": 10, "Novembro": 11, "Dezembro": 12
}

col1, col2, col3 = st.columns(3)

with col1:
    uf = st.selectbox("Estado (UF)", options=[
        'AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA',
        'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN',
        'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO'
    ], index=23) # Padrão SC

    populacao = st.number_input("População Total do Município", min_value=0, value=25000)

with col2:
    tipo_evento = st.selectbox("Tipo de Evento", options=[
        'Enxurradas', 'Inundações', 'Alagamentos', 'Tempestade Local/Convectiva',
        'Inundação', 'Vendaval / Ciclone', 'Estiagem', 'Seca', 'Granizo',
        'Geada', 'Incêndio Florestal', 'Deslizamentos', 'Outros'
    ], index=0)

    nome_mes = st.selectbox("Mês do Registro", options=list(meses_map.keys()), index=5)
    mes = meses_map[nome_mes]

    trimestre = (mes - 1) // 3 + 1
    st.info(f"🗓️ Trimestre estimado: **{trimestre}º Trimestre**")

with col3:
    hab_danificadas = st.number_input("Habitações Danificadas", min_value=0, value=0)
    hab_destruidas = st.number_input("Habitações Destruídas", min_value=0, value=0)
    infra_danificada = st.number_input("Obras de Infraestrutura Pública Danificadas", min_value=0, value=0)

# --- BOTÃO DE PREDIÇÃO E PROCESSAMENTO ---
st.markdown("---")

if st.button("🚀 Calcular Previsões", type="primary", use_container_width=True):
    try:
        # 1. Recuperar todas as colunas que o preprocessor/modelo espera
        try:
            colunas_esperadas = preprocessor.feature_names_in_
        except AttributeError:
            colunas_esperadas = preprocessor.steps[0][1].feature_names_in_

        # 2. Identificar colunas numéricas e categóricas do preprocessor
        col_categoricas = []
        col_numericas = []

        if hasattr(preprocessor, 'transformers_'):
            for name, trans, cols in preprocessor.transformers_:
                if name != 'remainder':
                    if any(c in str(trans).lower() for c in ['onehot', 'ordinal', 'encoder', 'categorical']):
                        col_categoricas.extend(cols)
                    else:
                        col_numericas.extend(cols)

        # 3. Preencher o dicionário com valores padrão corretos por tipo de coluna
        dados_dict = {}
        for col in colunas_esperadas:
            if col in col_categoricas or col in ['UF', 'Tipo_Evento']:
                dados_dict[col] = "Outros"  # Valor texto genérico seguro
            else:
                dados_dict[col] = 0.0      # Valor numérico float seguro

        # 4. Atualizar com as informações preenchidas no formulário
        dados_dict.update({
            'UF': str(uf),
            'Tipo_Evento': str(tipo_evento),
            'População': float(populacao),
            'Mes_Registro': float(mes),
            'Trimestre': float(trimestre),
            'DM_Unidades Habitacionais Danificadas': float(hab_danificadas),
            'DM_Unidades Habitacionais Destruídas': float(hab_destruidas),
            'DM_Obras de infraestrutura pública Danificadas': float(infra_danificada)
        })

        # 5. Criar DataFrame com dtypes corretos e na ordem exata esperada
        dados_entrada = pd.DataFrame([dados_dict])[colunas_esperadas]

        # Converter colunas numéricas explicitamente para float para evitar conflitos no numpy
        for col in dados_entrada.columns:
            if col not in ['UF', 'Tipo_Evento'] and col not in col_categoricas:
                dados_entrada[col] = pd.to_numeric(dados_entrada[col], errors='coerce').fillna(0.0)

        # 6. Transformar e Prever
        dados_processados = preprocessor.transform(dados_entrada)
        pred_risco_raw = modelo_clf.predict(dados_processados)[0]

        # Mapeamento do resultado numérico/ordinal para visualização
        mapa_risco = {
            0: "Baixo",
            1: "Médio",
            2: "Alto"
        }

        # Converte para inteiro e mapeia a classe sem usar bloco try/except
        if isinstance(pred_risco_raw, (int, float, np.integer, np.floating)):
            pred_risco = mapa_risco.get(int(pred_risco_raw), f"Classe ({pred_risco_raw})")
        else:
            pred_risco = str(pred_risco_raw)

        # Predição da População Afetada (Regressão)
        pred_pop_afetada_raw = modelo_reg.predict(dados_processados)[0]
        pred_pop_afetada = max(0, int(round(pred_pop_afetada_raw)))

        # Formatação no padrão de milhar brasileiro (ex: 1.250 pessoas)
        pop_formatada = f"{pred_pop_afetada:,}".replace(",", ".")

        # Exibição no Streamlit
        res_col1, res_col2 = st.columns(2)

        with res_col1:
            st.subheader("🔴 Nível de Risco Classificado")
            st.metric(label="Risco Estimado", value=pred_risco)

        with res_col2:
            st.subheader("👥 População Afetada Estimada")
            st.metric(label="Total de Pessoas Afetadas", value=f"{pop_formatada} pessoas")

    except Exception as err:
        st.error(f"Erro ao processar as previsões: {err}")

Overwriting app.py


In [30]:
from pyngrok import ngrok
import os

# 1. Configurar o seu Authtoken (substitua com o token do site)
ngrok.set_auth_token("3Eu0uGhV8Sw9rvvCGXlcwhNKIfj_81kRpesKtmutVx3YxxhhF")

# 2. Encerrar conexões anteriores
ngrok.kill()

# 3. Criar o túnel na porta 8501
public_url = ngrok.connect(8501)
print(f"🔗 Acesse sua aplicação sem erros aqui: {public_url}")

# 4. Rodar o Streamlit
!streamlit run app.py --server.port 8501

🔗 Acesse sua aplicação sem erros aqui: NgrokTunnel: "https://slideshow-litter-outspoken.ngrok-free.dev" -> "http://localhost:8501"


2026-09-15 14:48:47.982 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.10.187.233:8501

  Stopping...
  Stopping...
